<a href="https://colab.research.google.com/github/saeyeon055-hue/Bio-AI-Learning-Path/blob/main/Code/ANN%20programming%20with%20Class%20%EC%A0%95%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. XOR by PyTorch

In [ ]:
import numpy as np
import torch
import torch.nn as nn # neural net을 디자인하는 모듈
from sklearn.metrics import accuracy_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn # neural net을 디자인하는 모듈
import numpy as np
from sklearn.metrics import accuracy_score

# 데이터 읽기 함수
def load_dataset(file):
  data=np.loadtxt(file)
  print('DATA=',data)

  input_features = data[:, 0:-1]
  print('INPUT_FEATURES=',input_features)

  labels = np.reshape(data[:,-1],(4,1))
  print('LABELS=', labels)

  # PyTorch는 numpy array를 사용하지 않고 tensor의 자료 구조를 사용하기 때문에 torch.tensor로 변환해줘야한다.
  # .to(device)는 호출하는 함수에서 처리하도록 변경
  input_features = torch.tensor(input_features, dtype=torch.float)
  labels = torch.tensor(labels, dtype=torch.float)

  return (input_features, labels)

# 모델 평가 결과 계산을 위해 텐서를 리스트로 변환하는 함수
def tensor2list(input_tensor):
  return input_tensor.cpu().detach().numpy().tolist()

# 평가 수행 함수
def do_test(model, test_dataloader):

  # 평가 모드 셋팅
  model.eval()

  # Batch 별로 예측값과 정답을 저장할 리스트 초기화
  predicts, golds = [], []

  with torch.no_grad():

    for step, batch in enumerate(test_dataloader):

      # .cuda()를 통해 메모리에 업로드 (assuming model and data should be on CUDA if available)
      # This line implicitly assumes 'device' is 'cuda' if cuda is available,
      # or that the batch is already on the correct device.
      # For robustness, a 'device' argument could be passed to do_test and use t.to(device).
      batch = tuple(t.cuda() for t in batch)

      input_features, labels = batch
      hypothesis = model(input_features)
      logits = (hypothesis > 0.5).float()
      x = tensor2list(logits)
      y = tensor2list(labels)

      # 예측값과 정답을 리스트에 추가
      predicts.extend(x)
      golds.extend(y)

    print("PRED=",predicts)
    print("GOLD=",golds)
    print("Accuracy= {0:f}\n".format(accuracy_score(golds, predicts)))

In [ ]:
# GPU 사용 가능 여부 확인(torch.cuda.is_available)
if torch.cuda.is_available():
  device='cuda'
else:
  device='cpu'

input_features, labels = load_dataset('/content/drive/MyDrive/Colab Notebooks/기계학습/train.txt', device)

## NN 모델 만들기
model = nn.Sequential(
    nn.Linear(2, 2, bias=True), nn.Sigmoid(),
    nn.Linear(2, 1, bias=True), nn.Sigmoid().to(device)
)

# 이진분류 크로스에늩로피 비용 함수
loss_func = torch.nn.BCELoss().to(device)

# 옵티마이저 함수 (역전파 알고리즘을 수행할 함수)
optimizer = torch.optim.SGD(model.parameters(), lr=1)

# 학습 모드 셋팅
model.train()

DATA= [[0. 0. 0.]
 [0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 0.]]
INPUT_FEATURES= [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
LABELS= [[0.]
 [1.]
 [1.]
 [0.]]


Sequential(
  (0): Linear(in_features=2, out_features=2, bias=True)
  (1): Sigmoid()
  (2): Linear(in_features=2, out_features=1, bias=True)
  (3): Sigmoid()
)

In [ ]:
# 모델 학습
for epoch in range(1001):

  # 기울기 계산한 것들 초기화
  optimizer.zero_grad()

  # H(x) 계산: forward 연산
  hypothesis =model(input_features)

  # 비용 계산
  cost = loss_func(hypothesis, labels)

  # 역전파 수행
  cost.backward()
  optimizer.step()

  # 100 에폭마다 비용 출력
  if epoch % 100 == 0:
    print(epoch, cost.item())

0 0.6565458178520203
100 0.528904914855957
200 0.30812257528305054
300 0.11544300615787506
400 0.05984964966773987
500 0.03893052414059639
600 0.028485748916864395
700 0.02233029529452324
800 0.018304504454135895
900 0.015478736720979214
1000 0.013391830027103424


In [ ]:
# 평가 모드 셋팅 (학습 시 적용했던 드랍 아웃 여부 등을 비적용)
model.eval()

# 역전파를 적용하지 않도록 context manager 설정
with torch.no_grad():
  hypothesis = model(input_features)
  logits = (hypothesis > 0.5).float()
  predicts = tensor2list(logits)
  golds = tensor2list(labels)
  print('PRED=', predicts)
  print('GOLD]', golds)
  print('Accuracy : {0:f}'.format(accuracy_score(golds, predicts)))

PRED= [[0.0], [1.0], [1.0], [0.0]]
GOLD] [[0.0], [1.0], [1.0], [0.0]]
Accuracy : 1.000000


# 2. Wide ANN
> Hidden layer를 2 * 2에서 2 * 10으로 변경

> Widening은 선의 개수를 늘리는 효과

In [ ]:
if torch.cuda.is_available():
  device='cuda'
else:
  device='cpu'

input_features, labels = load_dataset('/content/drive/MyDrive/Colab Notebooks/기계학습/train.txt', device)

## NN 모델 만들기
model = nn.Sequential(
    nn.Linear(2, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 1, bias=True), nn.Sigmoid().to(device)
)

# 이진분류 크로스엔트로피 비용 함수
loss_func = torch.nn.BCELoss().to(device)

# 옵티마이저 함수 (역전파 알고리즘을 수행할 함수)
optimizer = torch.optim.SGD(model.parameters(), lr=1)

# 학습 모드 셋팅
model.train()

DATA= [[0. 0. 0.]
 [0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 0.]]
INPUT_FEATURES= [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
LABELS= [[0.]
 [1.]
 [1.]
 [0.]]


Sequential(
  (0): Linear(in_features=2, out_features=10, bias=True)
  (1): Sigmoid()
  (2): Linear(in_features=10, out_features=1, bias=True)
  (3): Sigmoid()
)

In [ ]:
# 모델 학습
for epoch in range(1001):

  # 기울기 계산한 것들 초기화
  optimizer.zero_grad()

  # H(x) 계산: forward 연산
  hypothesis =model(input_features)

  # 비용 계산
  cost = loss_func(hypothesis, labels)

  # 역전파 수행
  cost.backward()
  optimizer.step()

  # 100 에폭마다 비용 출력
  if epoch % 100 == 0:
    print(epoch, cost.item())

0 0.6932336091995239
100 0.6918482780456543
200 0.6853496432304382
300 0.6191054582595825
400 0.3300333023071289
500 0.0975780040025711
600 0.046307794749736786
700 0.029021821916103363
800 0.02079770341515541
900 0.01608039252460003
1000 0.01304871216416359


In [ ]:
# 평가 모드 셋팅 (학습 시 적용했던 드랍 아웃 여부 등을 비적용)
model.eval()

# 역전파를 적용하지 않도록 context manager 설정
with torch.no_grad():
  hypothesis = model(input_features)
  logits = (hypothesis > 0.5).float()
  predicts = tensor2list(logits)
  golds = tensor2list(labels)
  print('PRED=', predicts)
  print('GOLD]', golds)
  print('Accuracy : {0:f}'.format(accuracy_score(golds, predicts)))

PRED= [[0.0], [1.0], [1.0], [0.0]]
GOLD] [[0.0], [1.0], [1.0], [0.0]]
Accuracy : 1.000000


# 3. Shallow ANN
> Hidden layer를 없애고 single-layer perceptron으로 변경

In [ ]:
if torch.cuda.is_available():
  device='cuda'
else:
  device='cpu'

input_features, labels = load_dataset('/content/drive/MyDrive/Colab Notebooks/기계학습/train.txt', device)

## NN 모델 만들기
model = nn.Sequential(
    nn.Linear(2, 1, bias=True), nn.Sigmoid().to(device)
)

# 이진분류 크로스엔트로피 비용 함수
loss_func = torch.nn.BCELoss().to(device)

# 옵티마이저 함수 (역전파 알고리즘을 수행할 함수)
optimizer = torch.optim.SGD(model.parameters(), lr=1)

# 학습 모드 셋팅
model.train()

DATA= [[0. 0. 0.]
 [0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 0.]]
INPUT_FEATURES= [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
LABELS= [[0.]
 [1.]
 [1.]
 [0.]]


Sequential(
  (0): Linear(in_features=2, out_features=1, bias=True)
  (1): Sigmoid()
)

In [ ]:
# 모델 학습
for epoch in range(10001):

  # 기울기 계산한 것들 초기화
  optimizer.zero_grad()

  # H(x) 계산: forward 연산
  hypothesis =model(input_features)

  # 비용 계산
  cost = loss_func(hypothesis, labels)

  # 역전파 수행
  cost.backward()
  optimizer.step()

  # 1000 에폭마다 비용 출력
  if epoch % 1000 == 0:
    print(epoch, cost.item())

0 0.6931471824645996
1000 0.6931471824645996
2000 0.6931471824645996
3000 0.6931471824645996
4000 0.6931471824645996
5000 0.6931471824645996
6000 0.6931471824645996
7000 0.6931471824645996
8000 0.6931471824645996
9000 0.6931471824645996
10000 0.6931471824645996


In [ ]:
# 평가 모드 셋팅 (학습 시 적용했던 드랍 아웃 여부 등을 비적용)
model.eval()

# 역전파를 적용하지 않도록 context manager 설정
with torch.no_grad():
  hypothesis = model(input_features)
  logits = (hypothesis > 0.5).float()
  predicts = tensor2list(logits)
  golds = tensor2list(labels)
  print('PRED=', predicts)
  print('GOLD]', golds)
  print('Accuracy : {0:f}'.format(accuracy_score(golds, predicts)))

PRED= [[0.0], [0.0], [0.0], [0.0]]
GOLD] [[0.0], [1.0], [1.0], [0.0]]
Accuracy : 0.500000


> 학습 속도는 빠르지만 10,000 epoch를 수행해도 문제를 풀지 못함.

- 왜냐하면 single-layer perceptron은 linear separable problem 만 해결 가능하기 때문.

# 4. Deep ANN
> Hidden layer 층을 1개에서 2개로 변경

In [ ]:
if torch.cuda.is_available():
  device='cuda'
else:
  device='cpu'

input_features, labels = load_dataset('/content/drive/MyDrive/Colab Notebooks/기계학습/train.txt', device)

## NN 모델 만들기
model = nn.Sequential(
    nn.Linear(2, 2, bias=True), nn.Sigmoid(),
    nn.Linear(2, 2, bias=True), nn.Sigmoid(),
    nn.Linear(2, 1, bias=True), nn.Sigmoid()).to(device)

# 이진분류 크로스엔트피 비용 함수
loss_func = torch.nn.BCELoss().to(device)

# 옵티마이저 함수 (역전파 알고리즘을 수행할 함수)
optimizer = torch.optim.SGD(model.parameters(), lr=1)

# 학습 모드 셋팅
model.train()

DATA= [[0. 0. 0.]
 [0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 0.]]
INPUT_FEATURES= [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
LABELS= [[0.]
 [1.]
 [1.]
 [0.]]


Sequential(
  (0): Linear(in_features=2, out_features=2, bias=True)
  (1): Sigmoid()
  (2): Linear(in_features=2, out_features=2, bias=True)
  (3): Sigmoid()
  (4): Linear(in_features=2, out_features=1, bias=True)
  (5): Sigmoid()
)

In [ ]:
# 모델 학습
for epoch in range(4001):

  # 기울기 계산한 것들 초기화
  optimizer.zero_grad()

  # H(x) 계산: forward 연산
  hypothesis =model(input_features)

  # 비용 계산
  cost = loss_func(hypothesis, labels)

  # 역전파 수행
  cost.backward()
  optimizer.step()

  # 400 에폭마다 비용 출력
  if epoch % 400 == 0:
    print(epoch, cost.item())

0 0.6311635971069336
400 0.490317702293396
800 0.010756785050034523
1200 0.004827172961086035
1600 0.0030831736512482166
2000 0.0022559459321200848
2400 0.0017749276012182236
2800 0.0014610359212383628
3200 0.00124040013179183
3600 0.0010770109947770834
4000 0.0009511990356259048


In [ ]:
# 평가 모드 셋팅 (학습 시 적용했던 드랍 아웃 여부 등을 비적용)
model.eval()

# 역전파를 적용하지 않도록 context manager 설정
with torch.no_grad():
  hypothesis = model(input_features)
  logits = (hypothesis > 0.5).float()
  predicts = tensor2list(logits)
  golds = tensor2list(labels)
  print('PRED=', predicts)
  print('GOLD]', golds)
  print('Accuracy : {0:f}'.format(accuracy_score(golds, predicts)))

PRED= [[0.0], [1.0], [1.0], [0.0]]
GOLD] [[0.0], [1.0], [1.0], [0.0]]
Accuracy : 1.000000


> deeping은 선을 구부리는 효과

## 4-1. Hidden layer 층을 1개에서 7개로 변경

In [ ]:
if torch.cuda.is_available():
  device='cuda'
else:
  device='cpu'

input_features, labels = load_dataset('/content/drive/MyDrive/Colab Notebooks/기계학습/train.txt', device)

## NN 모델 만들기
model = nn.Sequential(
    nn.Linear(2, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 1, bias=True), nn.Sigmoid()).to(device)

# 이진분류 크로스엔트피 비용 함수
loss_func = torch.nn.BCELoss().to(device)

# 옵티마이저 함수 (역전파 알고리즘을 수행할 함수)
optimizer = torch.optim.SGD(model.parameters(), lr=1)

# 학습 모드 셋팅
model.train()

DATA= [[0. 0. 0.]
 [0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 0.]]
INPUT_FEATURES= [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
LABELS= [[0.]
 [1.]
 [1.]
 [0.]]


Sequential(
  (0): Linear(in_features=2, out_features=10, bias=True)
  (1): Sigmoid()
  (2): Linear(in_features=10, out_features=10, bias=True)
  (3): Sigmoid()
  (4): Linear(in_features=10, out_features=10, bias=True)
  (5): Sigmoid()
  (6): Linear(in_features=10, out_features=10, bias=True)
  (7): Sigmoid()
  (8): Linear(in_features=10, out_features=10, bias=True)
  (9): Sigmoid()
  (10): Linear(in_features=10, out_features=10, bias=True)
  (11): Sigmoid()
  (12): Linear(in_features=10, out_features=10, bias=True)
  (13): Sigmoid()
  (14): Linear(in_features=10, out_features=10, bias=True)
  (15): Sigmoid()
  (16): Linear(in_features=10, out_features=1, bias=True)
  (17): Sigmoid()
)

In [ ]:
# 모델 학습
for epoch in range(10001):

  # 기울기 계산한 것들 초기화
  optimizer.zero_grad()

  # H(x) 계산: forward 연산
  hypothesis =model(input_features)

  # 비용 계산
  cost = loss_func(hypothesis, labels)

  # 역전파 수행
  cost.backward()
  optimizer.step()

  # 1000 에폭마다 비용 출력
  if epoch % 1000 == 0:
    print(epoch, cost.item())

0 0.706458568572998
1000 0.6931471824645996
2000 0.6931471824645996
3000 0.6931471824645996
4000 0.6931471824645996
5000 0.6931471824645996
6000 0.6931471824645996
7000 0.6931471824645996
8000 0.6931471824645996
9000 0.6931471824645996
10000 0.6931471824645996


In [ ]:
# 평가 모드 셋팅 (학습 시 적용했던 드랍 아웃 여부 등을 비적용)
model.eval()

# 역전파를 적용하지 않도록 context manager 설정
with torch.no_grad():
  hypothesis = model(input_features)
  logits = (hypothesis > 0.5).float()
  predicts = tensor2list(logits)
  golds = tensor2list(labels)
  print('PRED=', predicts)
  print('GOLD]', golds)
  print('Accuracy : {0:f}'.format(accuracy_score(golds, predicts)))

PRED= [[1.0], [0.0], [0.0], [0.0]]
GOLD] [[0.0], [1.0], [1.0], [0.0]]
Accuracy : 0.250000


> 10,000 epoch를 돌려도 학습이 안됨
- 2차 winter season 원인

- 문제 해결을 위해 더 복잡한 층을 설계했는데 문제 해결이 되지 않음

> 원인: Vanishing Gradient
- 초기 hidden layer에서 아래로 잘 전파되다가, 깊어지니까 희미해지면서 전파가 되지 않게 됨.

- 결국 아래층에는 업데이트가 되지 않음.

# 5. Sigmoid to ReLU

In [ ]:
if torch.cuda.is_available():
  device='cuda'
else:
  device='cpu'

input_features, labels = load_dataset('/content/drive/MyDrive/Colab Notebooks/기계학습/train.txt', device)

## NN 모델 만들기 (마지막 layer는 0과 1 사이 값을 출력하도록 하기 위해서 signmoid 유지)
model = nn.Sequential(
    nn.Linear(2, 10, bias=True), nn.ReLU(),
    nn.Linear(10, 10, bias=True), nn.ReLU(),
    nn.Linear(10, 10, bias=True), nn.ReLU(),
    nn.Linear(10, 10, bias=True), nn.ReLU(),
    nn.Linear(10, 10, bias=True), nn.ReLU(),
    nn.Linear(10, 10, bias=True), nn.ReLU(),
    nn.Linear(10, 10, bias=True), nn.ReLU(),
    nn.Linear(10, 10, bias=True), nn.ReLU(),
    nn.Linear(10, 1, bias=True), nn.Sigmoid()).to(device)

# 이진분류 크로스엔트피 비용 함수
loss_func = torch.nn.BCELoss().to(device)

# 옵티마이저 함수 (역전파 알고리즘을 수행할 함수)
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)

# 학습 모드 셋팅
model.train()

DATA= [[0. 0. 0.]
 [0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 0.]]
INPUT_FEATURES= [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
LABELS= [[0.]
 [1.]
 [1.]
 [0.]]


Sequential(
  (0): Linear(in_features=2, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=10, bias=True)
  (3): ReLU()
  (4): Linear(in_features=10, out_features=10, bias=True)
  (5): ReLU()
  (6): Linear(in_features=10, out_features=10, bias=True)
  (7): ReLU()
  (8): Linear(in_features=10, out_features=10, bias=True)
  (9): ReLU()
  (10): Linear(in_features=10, out_features=10, bias=True)
  (11): ReLU()
  (12): Linear(in_features=10, out_features=10, bias=True)
  (13): ReLU()
  (14): Linear(in_features=10, out_features=10, bias=True)
  (15): ReLU()
  (16): Linear(in_features=10, out_features=1, bias=True)
  (17): Sigmoid()
)

In [ ]:
# 모델 학습
for epoch in range(3001):

  # 기울기 계산한 것들 초기화
  optimizer.zero_grad()

  # H(x) 계산: forward 연산
  hypothesis =model(input_features)

  # 비용 계산
  cost = loss_func(hypothesis, labels)

  # 역전파 수행
  cost.backward()
  optimizer.step()

  # 300 에폭마다 비용 출력
  if epoch % 300 == 0:
    print(epoch, cost.item())

0 0.6965017318725586
300 0.6930959224700928
600 0.69305819272995
900 0.6929864287376404
1200 0.6927822232246399
1500 0.6913377046585083
1800 0.003306519938632846
2100 0.0002116301329806447
2400 9.490660886513069e-05
2700 5.842854443471879e-05
3000 4.135034396313131e-05


In [ ]:
# 평가 모드 셋팅 (학습 시 적용했던 드랍 아웃 여부 등을 비적용)
model.eval()

# 역전파를 적용하지 않도록 context manager 설정
with torch.no_grad():
  hypothesis = model(input_features)
  logits = (hypothesis > 0.5).float()
  predicts = tensor2list(logits)
  golds = tensor2list(labels)
  print('PRED=', predicts)
  print('GOLD]', golds)
  print('Accuracy : {0:f}'.format(accuracy_score(golds, predicts)))

PRED= [[0.0], [1.0], [1.0], [0.0]]
GOLD] [[0.0], [1.0], [1.0], [0.0]]
Accuracy : 1.000000


# 6. Dropout by PyTorch
> 학습 과정 중에 지정된 비율로 임의의 연결을 끊음으로써 일반화 성능을 개선하는 방법

In [ ]:
if torch.cuda.is_available():
  device='cuda'
else:
  device='cpu'

input_features, labels = load_dataset('/content/drive/MyDrive/Colab Notebooks/기계학습/train.txt', device)

## NN 모델 만들기 (마지막 layer는 0과 1 사이 값을 출력하도록 하기 위해서 signmoid 유지)
model = nn.Sequential(
    nn.Linear(2, 10, bias=True), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(10, 10, bias=True), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(10, 10, bias=True), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(10, 10, bias=True), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(10, 10, bias=True), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(10, 10, bias=True), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(10, 10, bias=True), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(10, 1, bias=True), nn.Sigmoid()).to(device)

# 이진분류 크로스엔트피 비용 함수
loss_func = torch.nn.BCELoss().to(device)

# 옵티마이저 함수 (역전파 알고리즘을 수행할 함수)
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)

# 학습 모드 셋팅
model.train()

DATA= [[0. 0. 0.]
 [0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 0.]]
INPUT_FEATURES= [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
LABELS= [[0.]
 [1.]
 [1.]
 [0.]]


Sequential(
  (0): Linear(in_features=2, out_features=10, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=10, out_features=10, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.2, inplace=False)
  (6): Linear(in_features=10, out_features=10, bias=True)
  (7): ReLU()
  (8): Dropout(p=0.2, inplace=False)
  (9): Linear(in_features=10, out_features=10, bias=True)
  (10): ReLU()
  (11): Dropout(p=0.2, inplace=False)
  (12): Linear(in_features=10, out_features=10, bias=True)
  (13): ReLU()
  (14): Dropout(p=0.2, inplace=False)
  (15): Linear(in_features=10, out_features=10, bias=True)
  (16): ReLU()
  (17): Dropout(p=0.2, inplace=False)
  (18): Linear(in_features=10, out_features=10, bias=True)
  (19): ReLU()
  (20): Dropout(p=0.2, inplace=False)
  (21): Linear(in_features=10, out_features=1, bias=True)
  (22): Sigmoid()
)

In [ ]:
# 모델 학습
for epoch in range(2001):

  # 기울기 계산한 것들 초기화
  optimizer.zero_grad()

  # H(x) 계산: forward 연산
  hypothesis =model(input_features)

  # 비용 계산
  cost = loss_func(hypothesis, labels)

  # 역전파 수행
  cost.backward()
  optimizer.step()

  # 200 에폭마다 비용 출력
  if epoch % 200 == 0:
    print(epoch, cost.item())

0 0.6931410431861877
200 0.6930365562438965
400 0.692963719367981
600 0.6927772164344788
800 0.6919956803321838
1000 0.6730166673660278
1200 0.02142452262341976
1400 0.004531446378678083
1600 0.0021779348608106375
1800 0.0013584846165031195
2000 0.0009642943041399121


In [ ]:
# 평가 모드 셋팅 (학습 시에 적용했던 드랍 아웃 여부 등을 비적용)
model.eval()

# 역전파를 적용하지 않도록 context manager 설정
with torch.no_grad():
    hypothesis = model(input_features)
    logits = (hypothesis > 0.5).float()
    predicts = tensor2list(logits)
    golds = tensor2list(labels)
    print("PRED=",predicts)
    print("GOLD=",golds)
    print("Accuracy : {0:f}".format(accuracy_score(golds, predicts)))

PRED= [[0.0], [1.0], [1.0], [0.0]]
GOLD= [[0.0], [1.0], [1.0], [0.0]]
Accuracy : 1.000000


# 6. Residual Connection
> 가중치층을 우회하여 상위 층으로 직접 연결하는 것
> 추상화 정도(낮은 수준 추상화와 높은 수준 추상화)를 적절히 섞어주는 효과 -> 앙상블 효과를 통해 성능 개선

# 7. ANN Programming with Class

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import numpy as np
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
from torch.utils.data import (DataLoader, RandomSampler, TensorDataset)

class XOR(nn.Module):

  def __init__(self, config):
    super(XOR, self).__init__()

    # 입력층 노드 수
    self.inode = config["input_node"]
    # 은닉층 데이터 크기
    self.hnode = config["hidden_node"]
    # 출력층 노드 수: 분류해야 하는 레이블 수
    self.onode = config["output_node"]

    # 활성화 함수로 Sigmoid 사용
    self.activation = nn.Sigmoid()

    # 신경망 설계
    self.linear1 = nn.Linear(self.inode, self.hnode, bias=True)
    self.linear2 = nn.Linear(self.hnode, self.onode, bias=True)

  def forward(self, input_features):

    output1 = self.linear1(input_features)
    hypothesis1 = self.activation(output1)

    output2 = self.linear2(hypothesis1)
    hypothesis2 = self.activation(output2)

    return hypothesis2


## [3단계: ANN 모델 정의-객체화]

### 1. 상단 라이브러리 가져오기

운영체제(파일 경로 등) 제어 모듈
> import os

수치 계산, 행렬 연산용 모듈
> import numpy as np

6단계(평가)에서 정확도를 쉽게 계산해 주는 함수
> from sklearn.metrics import accuracy_score

파이토치 메인 모듈
> import torch

인공신경망(Neural Network)에 필요한 레이어, 활성화 함수 등이 담긴 모듈
> import torch.nn as nn

2단계(데이터 로더) 구성을 위한 모듈을
> from torch.utils.data import (DataLoader, RandomSampler, TensorDataset)

## 2.클래스 선언과 부로 클래스 상속
> class XOR(nn.Module):

- class XOR: 내가 만들 인공신경망의 이름을 XOR이라고 명명하고 객체화할 준비를 한다.

- (nn.Module): 파이토치로 신경망을 만들 때의 필수 규칙. nn.Module이라는 거대한 인공신경망 틀(부모 클래스)을 상속받아야만,파이토치가 제공하는 역전파(Backward)나 가중치 추적으 그대로 사용할수 있음.

## 3.생성자 함수 '__init__': 모델의 설계도 그리기
> 모델이 처음 생성될 때 딱 한 번 실행되며, 신경망에 필요한 방(레이어)들과 부품들을 정의.

config라는 딕셔너리 변수를 통해 외부에서 설정값을 받아온다.

super(XOR, self).__init__()
> 상속받은 부모 클래스(nn.Module)의 초기화 설정을 그대로 가져온다.

self.inode/hnode/onode['']
> config 딕셔너리에서 값을 꺼내와 변수에 저장.

> inode(입력층 노드 수), hnode(은닉층 노드 수), onode(출력층 노드 수)

self.activation
> 활성화 함수로 Sigmoid 사용(0~1 사이의 값으로 압축해 주는 함수)

self.linear1= nn.Linear(self.inode, self.hnode, bias=True)
: 입력층 -> 은닉층으로 가는 다리

self.linear2= nn.Linear(self.hnode, self.onode, bias=True)
: 은닉층 -> 출력층으로 가는 다리
> 신경망 설계(선형 결합 레이어 정의)

> nn.Linear(입력 데이터 크기, 출력 데이터 크기, 편향b 사용 여부)

## 4. forward 메서드: 데이터가 흘러가는 길 정의
> __init__에서 만든 방들을 바탕으로, 데이터(input_features)가 들어왔을 때 어떤 순서로 연산되어 최종 예측값으로 나가는지 연산 흐름 작성

입력 데이터를 첫 번째 선형층에 통과
> output1 = self.linear1(input_features)

그 결과를 시그모이드 활성화 함수에 넣어 비선형성 추가
> hypothesis1 = self.activation(output1)

은닉층의 결과물(hypothesis1)을 두 번째 선형층에 통과시킴
> output2 = self.linear2(hypothesis1)

시그모이드 함수에 넣어 최종 예측값을 만든다
> hypothesis2 = self.activation(output2)

최종 모델의 예측 결과를 반환
> return hypothesis2

In [3]:
# 데이터 읽기 함수
def load_dataset(file):
  data = np.loadtxt(file)
  print("DATA=",data)

  input_features = data[:,0:-1]
  print("INPUT_FEATURES=",input_features)

  labels = np.reshape(data[:,-1],(4,1))
  print("LABELS=",labels)

  input_features = torch.tensor(input_features, dtype=torch.float)
  labels = torch.tensor(labels, dtype=torch.float)

  return (input_features, labels)

## [1단게: 데이터 로드/전처리] 및 [2단계: 데이터셋 구성]

### 1. 파일읽기
텍스트 파일을 읽어 NumPy 행렬로 변환
> def load_dataset(file):

> data = np.loadtxt(file)


### 2. 문제(Features) 슬라이싱
> 전체 데이터에서 정답을 제외한 입력값(x)만 빼내는 과정


NumPy 슬라이싱: [모든 행, 처음부터 마지막 열 직전까지]
> input_features = data[:, 0:-1]

> print("INPUT_FEATURES=", input_features)

### 3. 정답(Lables) 슬라이싱 및 모양 변형

data[:, -1]로 맨 마지막 열(정답)만 가져오면 일렬로 늘어진 형태(가로 벡터)가 된다.

이를 AI가 계산하기 좋게 4행 1열의 세로 형태로 명시적으로 모양을 변경(reshape)한다.
> labels = np.reshape(data[:, -1], (4, 1))

### 4. 파이토치 텐서로 변환
> NumPy 행렬은 파이토치 모델이 인식하지 못하고, GPU 연산도 불가능

torch.tensor()를 사용해 NumPy 데이터를 파이토치의 데이터 단위인 '텐서(Tensor)'로 변환.

딥러닝 연산에서 가장 표준적으로 사용하는 실수형 타입(torch.float 또는 float32)으로 지정.
> input_features = torch.tensor(input_features, dtype=torch.float)

> labels = torch.tensor(labels, dtype=torch.float)

변환된 문제집(인풋)과 해설지(레이블)를 튜플 형태로 반환.
> return (input_features, labels)

In [4]:
output_dir = '/content/output'
input_data = [] # Placeholder: You should replace this with your actual input data

config = {'mode': 'test',
          'model_name':'epoch_{0:d}.pt'.format(1000),
          'output_dir':output_dir,
          'input_data':input_data,
          'input_node':2,
          'hidden_node':10,
          'output_node':1, # Corrected typo from 'output_nude' to 'output_node'
          'learn_rate':1,
          'batch_size':4,
          'epoch':1000,
          }

### batch_size
XOR 같은 경우 데이터가 작은데, 데이터가 큰 경우 일부를 끊어서 올려야됨.

몇개로 끊어서 올릴 것인지
batch_size:n
=> n개로 끊는다

backpropagation할때 batch 단위로 진행하기 때문에 잘 디자인해야함.

너무 작게하면 오래걸리거 너무 크게하면 하나하나의 오류가 잘 반영되지 않는다.

In [5]:
# XOR Class

class XOR(nn.Module):

  def __init__(self, config):
    super(XOR, self).__init__()

    # 입력층 노드 수
    self.inode = config['input_node']
    # 은닉층 데이터 크기
    self.hnode = config['hidden_node']
    # 출력층 노드 수: 분류해야 하는 레이블 수
    self.onode = config['output_node']

    # 활성화 함수로 Sigmoid 사용
    self.activation = nn.Sigmoid()

    # 신경망 설계
    self.linear1 = nn.Linear(self.inode, self.hnode, bias=True)
    self.linear2 = nn.Linear(self.hnode, self.onode, bias=True)

  def forward(self, input_features):

    # hidden layer
    output1 = self.linear1(input_features)
    hypothesis1 = self.activation(output1)

    # output layer
    output2 = self.linear2(hypothesis1)
    hypothesis2 = self.activation(output2)

    return hypothesis2

> ANN 모듈을 설계할 때(class를 설계할 때) 생성자와 forward를 override 해야된다.

> 생성자: '__init__'

  - 이 class에서 사용할 attribute를 지정

> forward

  - 신경망을 실제로 구조화하는 것.(hypothesis 만들기)

In [9]:
# Training 함수
def train(config):

  # 모델 생성 (GPU 사용)
  model = XOR(config).cuda()

  # 데이터 읽기
  (input_features, labels) = load_dataset(config['input_data'])

  # TensorDataset/DataLoader를 통해 배치(batch) 단위로 데이터를 나누고 셔플(shuffle)
  train_featuers = TensorDataset(input_features, labels)
  train_dataloader = DataLoader(train_featuers, shuffle=True, batch_size=config['batch_size'])


  # 이진분류 크로스엔트로피 비용 함수
  loss_func = nn.BCELoss()
  # 옵티마이저 함수 (역전파 알고리즘 수행할 함수)
  optimizer = torch.optim.SGD(model.parameters(), lr=config['learn_rate'])


## [2단계: DataLoader구성], [3단계: 모델 객체화], [4단계: 손실함수 & 옵티마이저 설정]

### 1. 모델 객체화 및 GPU 탑재[3단계]
XOR 설계도로 진짜 모델을 만든다.(객체화)

맨 뒤에 '.cuda()'를 붙여서 이 모델의 연산을 GPU(CUA)에서 하도록 지정

> model = XOR(config).cuda()

### 2. 데이터 로드 및 묶기 [2단계]
이전에 만든 load_dataset 함수를 써서 텐서 형태의 문제(features)와 정답(labels)을 가져온다.
> (input_features, labels) = load_dataset(config['input_data'])

TensorDataset: 문제집과 해설지를 한 권의 '세트'로 단단히 묶어주는 역할
> train_features = TensorDataset(input_features, labels)

### 3. 데이터로더 배치 설정 [2단계]
DataLoader: 묶인 데이터를 설정한 배치 크기만큼 쪼개서 모델에 공금해 주는 역할
>  train_dataloader = DataLoader(train_featuers, shuffle=True, batch_size=config['batch_size'])

### 4. 손실함수 및 옵티마이저 설정 [4단계]

정답이 0 또는 1인 '이진 분류(Binary Classification)' 문제에서 표준적으로 사용하는 손실함수
> loss_func = nn.BCELoss()

torch.optim.SGD(): 가장 기본적인 최적화 알고리즘인 확률적 경사하강법(SGD)을 사용

model.parameters(): "XOR 모델 내부의 모든 가중치(W)와 편향(b)을 옵티마이저에게 넘겨줄 테니 네가 관리해라"라는 뜻

lr(Learning Rate): 가중치를 한 번에 얼마나 크게 수정할지 결정하는 보폭(학습률)
> optimizer = torch.optim.SGD(model.parameters(), lr=config['learn_rate'])

In [12]:
# Training 함수
def train(config):

  # 모델 생성 (GPU 사용)
  model = XOR(config).cuda()

  # 데이터 읽기
  (input_features, labels) = load_dataset(config['input_data'])

  # TensorDataset/DataLoader를 통해 배치(batch) 단위로 데이터를 나누고 셔플(shuffle)
  train_featuers = TensorDataset(input_features, labels)
  train_dataloader = DataLoader(train_featuers, shuffle=True, batch_size=config['batch_size'])


  # 이진분류 크로스엔트로피 비용 함수
  loss_func = nn.BCELoss()
  # 옵티마이저 함수 (역전파 알고리즘 수행할 함수)
  optimizer = torch.optim.SGD(model.parameters(), lr=config['learn_rate'])

  # 여기에 model은 def train(config) 내부에서 model=XOR(config).cuda()라고 모델의 정의 했기 때문에, 그 모델 변수는 train 함수 안에서만 사용 가능. 따라서 들여쓰기 해야함.
  for epoch in range(config["epoch"]+1):

    # 학습 모드 셋팅
    model.train()

    # epoch 마다 평균 비용을 저장하기 위한 리스트
    costs = []

    for (step, batch) in enumerate(train_dataloader):

      # batch = (input_features[step], labels[step])*batch_size
      # .cuda()를 통해 메모리에 업로드
      batch = tuple(t.cuda() for t in batch)

      # 각 feature 저장
      input_features, labels = batch

      # 역전파 변화도 초기화
      # .backward() 호출 시, 변화도 버퍼에 데이터가 계속 누적한 것을 초기화
      optimizer.zero_grad()

      # H(X) 계산: forward 연산
      hypothesis = model(input_features)
      # 비용 계산
      cost = loss_func(hypothesis, labels)
      # 역전파 수행
      cost.backward()
      optimizer.step()

      # 현재 batch의 스텝 별 loss 저장
      costs.append(cost.data.item())

    # 100 에폭마다 평균 loss 출력하고 모델을 저장
    if epoch%100 == 0:
      print("Average Loss= {0:f}".format(np.mean(costs)))
      torch.save(model.state_dict(), os.path.join(config["output_dir"], "epoch_{0:d}.pt".format(epoch)))
      do_test(model, train_dataloader)


## [5단계: 모델 학습 반복문 (Training Loop)]

### 1. 에포크(Epoch) 반복문과 학습 모드 설정

model.train(): 모델을 '학습 모드'로 설정. (드롭아웃이나 배치 정규화 같은 기법들이 학습용으로 켜짐.)
> model.train()

에포크마다 쪼개진 배치들의 오차(loss)를 모아서 평균을 내기 위해 빈 리스트 준비
> costs =[]

### 2. 배치(Batch) 단위 데이터 꺼내기 및 GPU 탑재
train_dataloader에서 배치 크기만큼 데이터를 쏙쏙 꺼낸다. (step은 몇 번째 배치인지 세어주는 인덱스)
> for (step, batch) in enumerate(train_dataloader):

꺼내온 데이터(문제, 정답)를 GPU 메모리로 보낸다. (.cuda() 처리)

모델이 GPU에 있으므로 데이터도 반드시 GPU에 있어야 연산 가능
>  batch = tuple(t.cuda() for t in batch)

튜플로 묶여있던 데이터를 각각 'input_features(문제)'와 'labels(정답)' 변수에 나누어 담는다.
> input_features, labels = batch

### 3. 학습 과정
1) 기울기 리셋: 이전 배치에서 계산했던 기울기(Gradient) 지우기. 파이토치는 기본적으로 기울기를 누적시키는 성질이 있어서 매번 리셋해야 함.
> optimizer.zero_grad()

2) 순전파(forward): 모델에 문제를 집어넣어 예측값(hypothesis)을 얻는다.
> hyopthesis = model(input_features)

3) 오차 계산: 예측값과 실제 정답을 비교해 얼마나 틀렸는지 점수(cost)를 매긴다.
cost = loss_func(hypothesis, labels)

4) 역전파 및 수정: 오차를 바탕으로 가중치들이 얼마나 수정되어야 하는지 기울기를 구하고(backward), 그 값을 토대로 실제 모델의 가중치(W)와 편향(b)을 한 걸음 업데이트(step)
> cost.backward()

> optimizer.step()

5) 오차 숫자(수치값)만 costs 리스트에 저장
> cost.append(cost.data.item())

### 4. 중간 점검 및 모델 저장
100 에포크마다 한 번씩 학습이 잘 되고 있는지 출력하고 저장
> if epoch%100 == 0:

현재 에포크에서 모든 배치들이 냈던 오차들의 평균을 계산해 화면에 출력
> print('Average Loss= {0:f}', format(np.mean(costs)))

현재까지 학습된 모델의 가중치만 꺼내서 지정된 경로에 '.pt'파일로 저장
> torch.save(model.state_dic(), os.path.join(config['output_dir'], 'epoch_{0:d}.pt, format(epoch)))

100 에포크마다 모델의 성능을 테스트하는 사용자 정의 함수 호출
> do_test(model, train_dataloader)

In [13]:
# Test함수
# 모델 평가 함수
def test(config):

  model = XOR(config).cuda()

  # 저장된 모델 가중치 로드
  model.load_state_dict(torch.load(os.path.join(config['output_dir'], config['model_name'])))

  # 데이터 load
  (features, labels) = load_dataset(config['input_data'])

  test_features = TensorDataset(features, labels)
  test_dataloader = DataLoader(test_features, shuffle=False, batch_size=config['batch_size'])

  do_test(model, test_dataloader)


## [6단계: 모델 검증 및 평가], [7단계: 저장된 모델 불러오기]를 담당하는 test 함수

### 1. 모델 생성 및 GPU 탑재
만든 XOR 설계도 모델 뼈대를 만든다.
> model=XOR(config).cuda()

### 2. 저장된 가중치 합체하기 [7단계]
torch.load(): 하드디스크에 저장된 가중치 파일('.pt')을 메모리로 읽어옴.

model.load_state_dict(): 읽어온 가중치들을 방금 만든 뼈대 모델에 합체.
>  model.load_state_dict(torch.load(os.path.join(config['output_dir'], config['model_name'])))

### 3. 평가용 데이터셋 세팅
평가에 사용할 데이터를 텐서 형태로 읽기
> (features, labels) = load_dataset(config['input_data'])

학습 때와 마찬가지로 문제집과 해설지를 묶는다.
> test_features = TensorDataset(features, labels)

### 4. 실전 모의고사 치르기 [6단계]
데이터로더와 모델을 실제 채점 및 평가 수행하는 'do_test' 함수로 넘김.
> do_test(model, test_dataloader)


In [22]:
# 모델 평가 결과 계산을 위해 텐서를 리스트로 변환하는 함수
def tensor2list(input_tensor):
    return input_tensor.cpu().detach().numpy().tolist()

# 평가 수행 함수
def do_test(model, test_dataloader):

  # 평가 모드 셋팅
  model.eval()

  # Batch 별로 예측값과 정답을 저장할 리스트 초기화
  predicts, golds = [], []

  with torch.no_grad():

    for step, batch in enumerate(test_dataloader):

      # .cuda()를 통해 메모리에 업로드
      batch = tuple(t.cuda() for t in batch)

      input_features, labels = batch
      hypothesis = model(input_features)
      logits = (hypothesis > 0.5).float()
      x = tensor2list(logits)
      y = tensor2list(labels)

      # 예측값과 정답을 리스트에 추가
      predicts.extend(x)
      golds.extend(y)

    print("PRED=",predicts)
    print("GOLD=",golds)
    print("Accuracy= {0:f}\n".format(accuracy_score(golds, predicts)))

## [6단계: 모델 검증 및 평가]

### 1. tensor2list: 파이토치 데이터를 순수 파이썬 리스트로 바꾸기
> def tensor2list(input_tensor):

>    return input_tensor.cpu().detach().numpy().tolist()

### 2. 평가 모드 전환과 기록 장치 끄기
def do_test(model, test_dataloader):

1) model.eval(): 모델을 '평가(시험) 모드'로 전환. 학습 때만 쓰던 임의의 가공 기법(드롭아웃 등)을 끄고 실전 시험 모드로 대기.
> model.eval()

2) 모델의 예측값과 실제 정답을 모아둘 빈 주머니 생성
> predicts, golds = [], []

3) 평가할 때는 공부(역전파)를 하면 안 되므로, 기울기를 계산하지 못하게 명령
> with torch.no_grad():

### 3. 시험 문제 풀기 및 이진 분류 판단 (Logits)
시그모이드를 통과한 확률값이 0.5보다 크면 True(1.0), 작으면 False(0.0)로 변환.

인공신경망의 점수를 확정적인 정답(0 또는 1)으로 바꾸는 과정
> logits = (hypothesis > 0.5).float()

텐서 데이터를 위에서 만든 함수를 통해 파이썬 리스트로 변환.
> x = tensor2list(logits)

> y = tensor2list(labels)

.append()가 아니라 .extend()를 사용해 리스트 안에 값들을 낱개로 연달아 이어 붙임.

- extend를 사용한 이유: 만약 x가 [1,0]일 때 append(x)를 쓰면 [[1,0]]처럼 리스트 안에 리스트가 중첩되어 들어가 오류 발생시킴

- extend(x)를 쓰면 [1,0]처럼 알맹이만 풀어서 일렬로 넣어주기 때문에 딥러닝에서 배치를 누적할 때는 무조건 extend를 쓴다.

> predicts.extend(x)

> golds.extend(y)

### [5단계] 학습 실행

In [29]:
# Training in Main
if(__name__=="__main__"):

    root_dir = "/content/drive/MyDrive/Colab Notebooks/기계학습/ann/xor/output"
    output_dir = os.path.join(root_dir, "output")
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    input_data = "/content/drive/MyDrive/Colab Notebooks/기계학습/train.txt"

    config = {"mode": "train",
              "model_name":"epoch_{0:d}.pt".format(1000),
              "output_dir":output_dir,
              "input_data":input_data,
              "input_node":2,
              "hidden_node":10,
              "output_node":1,
              "learn_rate":1,
              "batch_size":4,
              "epoch":1000,
              }

    if(config["mode"] == "train"):
        train(config)
    else:
        test(config)

DATA= [[0. 0. 0.]
 [0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 0.]]
INPUT_FEATURES= [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
LABELS= [[0.]
 [1.]
 [1.]
 [0.]]
Average Loss= 0.698586
PRED= [[1.0], [1.0], [1.0], [0.0]]
GOLD= [[1.0], [0.0], [0.0], [1.0]]
Accuracy= 0.250000

Average Loss= 0.678532
PRED= [[1.0], [0.0], [1.0], [1.0]]
GOLD= [[1.0], [0.0], [0.0], [1.0]]
Accuracy= 0.750000

Average Loss= 0.497313
PRED= [[1.0], [0.0], [1.0], [0.0]]
GOLD= [[1.0], [0.0], [1.0], [0.0]]
Accuracy= 1.000000

Average Loss= 0.135675
PRED= [[0.0], [1.0], [0.0], [1.0]]
GOLD= [[0.0], [1.0], [0.0], [1.0]]
Accuracy= 1.000000

Average Loss= 0.055228
PRED= [[1.0], [0.0], [1.0], [0.0]]
GOLD= [[1.0], [0.0], [1.0], [0.0]]
Accuracy= 1.000000

Average Loss= 0.032346
PRED= [[1.0], [0.0], [1.0], [0.0]]
GOLD= [[1.0], [0.0], [1.0], [0.0]]
Accuracy= 1.000000

Average Loss= 0.022341
PRED= [[1.0], [0.0], [1.0], [0.0]]
GOLD= [[1.0], [0.0], [1.0], [0.0]]
Accuracy= 1.000000

Average Loss= 0.016875
PRED= [[0.0], [1.0], [1.0], [0.0]]
GOLD= [[0.

### [6단계] 평가 실행

In [30]:
# Test in Main
if(__name__=="__main__"):

    root_dir = "/content/drive/MyDrive/Colab Notebooks/기계학습/ann/xor/output"
    output_dir = os.path.join(root_dir, "output")
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    input_data = "/content/drive/MyDrive/Colab Notebooks/기계학습/train.txt"

    config = {"mode": "test",
              "model_name":"epoch_{0:d}.pt".format(1000),
              "output_dir":output_dir,
              "input_data":input_data,
              "input_node":2,
              "hidden_node":10,
              "output_node":1,
              "learn_rate":1,
              "batch_size":4,
              "epoch":1000,
              }

    if(config["mode"] == "train"):
        train(config)
    else:
        test(config)

DATA= [[0. 0. 0.]
 [0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 0.]]
INPUT_FEATURES= [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
LABELS= [[0.]
 [1.]
 [1.]
 [0.]]
PRED= [[0.0], [1.0], [1.0], [0.0]]
GOLD= [[0.0], [1.0], [1.0], [0.0]]
Accuracy= 1.000000

